<a href="https://colab.research.google.com/github/triman1905/100-days-of-machine-learning/blob/main/final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub, os
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
DATA_DIR = os.path.join(path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)")
print("✅ Path:", DATA_DIR)
print("✅ Exists:", os.path.exists(DATA_DIR))

In [ ]:
from datasets import load_dataset
dataset = load_dataset("imagefolder", data_dir=DATA_DIR)
print(dataset)

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
model_name = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)
labels = dataset["train"].features["label"].names
print("✅ Labels:", len(labels))

model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label={i: l for i, l in enumerate(labels)},
    label2id={l: i for i, l in enumerate(labels)},
    ignore_mismatched_sizes=True
)

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(4000))

def transform(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

dataset = dataset.with_transform(transform)
print("✅ Transform applied")

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="epoch",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
)

trainer.train()
print("✅ TRAINING DONE")

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick any plant leaf image from your PC

In [ ]:
from PIL import Image
import torch

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(image, return_tensors="pt")

    # move to GPU if available
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred_idx = logits.argmax(-1).item()
    confidence = torch.softmax(logits, dim=1).max().item()

    print("🌿 Prediction:", model.config.id2label[pred_idx])
    print("📊 Confidence:", f"{confidence * 100:.2f}%")

# replace with your uploaded filename
predict("/content/test2.jpeg")